In [ ]:
# !conda install nltk scikit-learn numpy pandas matplotlib -y # anaconda python 3 
!pip install nltk scikit-learn numpy pandas matplotlib # osnovni python

In [45]:
 # preuzeti podatke za nltk
import nltk
nltk.download('punkt_tab')
nltk.download('brown')
nltk.download('universal_tagset')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\dseve\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\dseve\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     C:\Users\dseve\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\universal_tagset.zip.


True

# Kolokvij 1: priprema

## Zadatak 1: parsiranje rječnika

U mapi `data\rjecnik.txt` dano vam je popis riječi sa gramatičkim i semantičkim obilježjima.  Iz teksta izvući samo imenice sa opisom i gramatičkim obilježjima i spremiti u JSON datoteku prema sljedećem formatu
  
  
Npr. tekst:
```
gȁd	
im. m. 〈G gȁda; mn. N gȁdovi, G gȁdōvā〉 podmukla, pokvarena, nepoštena osoba; sin. podlac, pokvarenjak
```

treba parsirati u JSON datoteku

```json
gȁd.json
[
  {
    "lema": "gȁd",
    "pos": "im",
    "gender" : "m",
    "inflection": "G gȁda; mn. N gȁdovi, G gȁdōvā",           
    "definition": "podmukla, pokvarena, nepoštena osoba; sin. podlac, pokvarenjak"
  }
  ]
```




In [9]:
import re
import json
from pprint import pprint

with open('data/ocr.txt', 'r', encoding='utf8') as ocr:
    content = ocr.read()

    entries = re.split(r'\n\n', content)

    regex = r'(?P<lemma>\w+)\s+(?P<pos>im.)\s+(?P<gender>.*)\s+〈(?P<inflection>.*)〉(?P<definition>.*)'    

    for i,data in enumerate(entries):
        print(f'\n\npodatak {i}: ', data) 
        lex = {}
        mObj = re.match(regex, data, re.MULTILINE | re.DOTALL)


        if mObj:
            print('\n**Pronasao uzorak: ', end=' ')
            print(mObj.groupdict())
            lex['lemma'], lex['pos'], lex['gender'], lex['inflection'], lex['definition'] = mObj.group('lemma'), mObj.group("pos"), mObj.group("gender"), mObj.group("inflection"), mObj.group("definition")
            
            jsonObj = json.dumps(lex, ensure_ascii=False, indent=4)

            with open(f"data/{lex['lemma']}.json", "w", encoding='utf8') as outfile:
                outfile.write(jsonObj)
        else: 
            print('Nije imenica')



podatak 0:  gácati	
gl. nesvrš. neprijel. 〈prez. 1. l. jd. gȃcām, 3. l. mn. gácajū, imp. gȃcāj, aor. gácah, imperf. gȃcāh, prid. r. gácao〉 hodati ili gaziti po žitkome ili mokrome tlu [~ po blatu]
Nije imenica


podatak 1:  gàčac	
im. m. 〈G gàčca; mn. N gàčci, G gȁčācā〉 zool. ptica iz roda gavrana, sjajna crnoplava perja i tanka sivocrna kljuna

**Pronasao uzorak:  {'lemma': 'gàčac', 'pos': 'im.', 'gender': 'm.', 'inflection': 'G gàčca; mn. N gàčci, G gȁčācā', 'definition': ' zool. ptica iz roda gavrana, sjajna crnoplava perja i tanka sivocrna kljuna'}


podatak 2:  gȁd	
im. m. 〈G gȁda; mn. N gȁdovi, G gȁdōvā〉 podmukla, pokvarena, nepoštena osoba; sin. podlac, pokvarenjak

**Pronasao uzorak:  {'lemma': 'gȁd', 'pos': 'im.', 'gender': 'm.', 'inflection': 'G gȁda; mn. N gȁdovi, G gȁdōvā', 'definition': ' podmukla, pokvarena, nepoštena osoba; sin. podlac, pokvarenjak'}


podatak 3:  gȁdan	
prid. 〈G gàdna; odr. gàdnī, G gàdnōg(a); ž. gàdna, s. gȁdno; komp. gàdnijī〉 1. koji izaziva gađenje

## Zadatak 2

U ovom zadatku potrebno je izgraditi n-gramski model za tekstove HR jezika koji su dani u prilogu. Tekstovi su dani u .txt formatu.

Učinite sljedeće:
  1. Izgradite skup za treniranje na kojem ćete naučiti model (80% rečenica korpusa). Evaluirajte model uz pomoć mjere perpleksnosti na skupu za testiranje (20%). Na tekstu primijenite neku od normalizacijskih  tehnika kako bi dobili što bolju perpleksnost. 
  2. Pored MLE procjenitelja, koristite barem metodu zaglađivanja i interpolacije. Odaberite onu metodu koja daje najbolju perpleksnost. 
  3. Generirajte nekoliko rečenica iz najboljeg modela



In [10]:
import io

with io.open('data/kafka-preobrazaj3.txt', encoding='utf8') as fin:
    text = fin.read()

text[:50]

'Franz Kafka\n\nPreobražaj\n\ns njemačkog preveo\n\nZlatk'

In [12]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.util import bigrams, ngrams
from nltk.lm.models import MLE, Laplace


In [ ]:
# ucitaj hr učestale riječi koje želimo izbaciti iz teksta
with open('data/hr_stopwords.txt') as f:
    stopWordsCro= f.read()

print(stopWordsCro)

'a', 'ako', 'ali', 'bi', 'bih', 'bila', 'bili', 'bilo', 'bio', 'bismo', 'biste', 'biti', 'bumo', 'da', 'do', 'duÅ¾', 'ga', 'hoÄ‡e', 'hoÄ‡emo', 'hoÄ‡ete', 'hoÄ‡eÅ¡', 'hoÄ‡u', 'i', 'iako', 'ih', 'ili', 'iz', 'ja', 'je', 'jedna', 'jedne', 'jedno', 'jer', 'jesam', 'jesi', 'jesmo', 'jest', 'jeste', 'jesu', 'jim', 'joj', 'joÅ¡', 'ju', 'kada', 'kako', 'kao', 'koja', 'koje', 'koji', 'kojima', 'koju', 'kroz', 'li', 'me', 'mene', 'meni', 'mi', 'mimo', 'moj', 'moja', 'moje', 'mu', 'na', 'nad', 'nakon', 'nam', 'nama', 'nas', 'naÅ¡', 'naÅ¡a', 'naÅ¡e', 'naÅ¡eg', 'ne', 'nego', 'neka', 'neki', 'nekog', 'neku', 'nema', 'netko', 'neÄ‡e', 'neÄ‡emo', 'neÄ‡ete', 'neÄ‡eÅ¡', 'neÄ‡u', 'neÅ¡to', 'ni', 'nije', 'nikoga', 'nikoje', 'nikoju', 'nisam', 'nisi', 'nismo', 'niste', 'nisu', 'njega', 'njegov', 'njegova', 'njegovo', 'njemu', 'njezin', 'njezina', 'njezino', 'njih', 'njihov', 'njihova', 'njihovo', 'njim', 'njima', 'njoj', 'nju', 'no', 'o', 'od', 'odmah', 'on', 'ona', 'oni', 'ono', 'ova', 'pa', 'pak', 'po', 

In [14]:
# tokenizacija i normalizacija teksta
preprocess_text = [list(map(str.lower, word_tokenize(sent))) for sent in sent_tokenize(text)]
tokenized_text = []

# filtriraj riječi koji nisu u stopWordsCro
for i in preprocess_text:
    tokenized_text.append([w for w in i if not w in stopWordsCro])

# provjera
tokenized_text[:10]

[['franz',
  'kafka',
  'preobražaj',
  'njemačkog',
  'preveo',
  'zlatko',
  'gorjan',
  'projekt',
  'sufinancirala',
  'europska',
  'unija',
  'europskog',
  'socijalnog',
  'fonda'],
 ['više',
  'informacija',
  'eu',
  'fondovima',
  'možete',
  'naći',
  'web',
  'stranicama',
  'ministarstva',
  'regionalnoga',
  'razvoja',
  'fondova',
  'europske',
  'unije',
  'www.strukturnifondovi.hr',
  'sadržaj',
  'ovog',
  'materijala',
  'isključiva',
  'odgovornost',
  'hrvatske',
  'akademske',
  'istraživačke',
  'mreže',
  '–',
  'carnet'],
 ['sadržaj',
  'autoru',
  'franz',
  'kafka',
  'čitanja',
  'preobražaj',
  'ii',
  'iii',
  'metodički',
  'instrumentarij',
  'poticaji',
  'daljnji',
  'rad',
  'kviz',
  'franz',
  'kafka',
  'prag',
  '3.',
  'srpnja',
  '1883'],
 ['–', 'sanatorij', 'kierling', 'kraj', 'beča', '3.', 'lipnja', '1924'],
 ['franz',
  'kafka',
  'najneobičnijih',
  'figura',
  'književnosti',
  '20.',
  'stoljeća'],
 ['rođen',
  'obitelji',
  'židovskog',
 

In [15]:
# podjela podataka na trening i test skup
# ručna podjela podataka na 80% trening i 20% test skup
size = int(0.8 * len(tokenized_text))
trainSet = tokenized_text[:size]
testSet = tokenized_text[size:]

In [16]:
# nadopuna n-grama sa paddingom
n = 4
train_data, padded_sents = padded_everygram_pipeline(n, trainSet)

In [17]:
# treniranje i evaluacija modela
mle = MLE(n)
mle.fit(train_data, padded_sents)

In [18]:
# kolika je perpleksnost modela?
mle.perplexity(testSet) # inf jer imamo n-grame koji nisu vidjeni u trening skupu!

inf

In [19]:
# Laplace-ov model
train_data, padded_sents = padded_everygram_pipeline(n, trainSet)

laplace = Laplace(n)
laplace.fit(train_data, padded_sents)

In [20]:
laplace.perplexity(testSet)

4954.208921739823

In [21]:
# generiranje rečenica
# pretvori listu tokena u rečenicu
from nltk.tokenize.treebank import TreebankWordDetokenizer

def generate_sent(model, num_words, random_seed=42):
    content = []
    for token in model.generate(num_words, random_seed=random_seed):
        if token == '<s>':
            continue
        if token == '</s>':
            break
        content.append(token)
    return TreebankWordDetokenizer().detokenize(content)

In [31]:
generate_sent(laplace, 10, random_seed=1000)

'sobu stolicu opet dogurala točno onog istog mjesta čak otada'

# Zadatak 3: POS označavatelj

Implementirajte POS označavatelj koristeći n-gramski Bayesov model na primjeru Brown-ovog korpusa žanra `news`.


In [22]:

# Setup za POS označavatelj
import nltk
import random
from nltk.corpus import brown
from nltk import RegexpTagger

# Setup Brown corpus data
brown_sents = brown.sents(categories='news')  # Get news sentences
brown_tagged_sents = [sentence for sentence in nltk.corpus.brown.tagged_sents(categories='news', tagset='universal')]

# Split data 9:1 for training and testing
size = int(0.9 * len(brown_tagged_sents))
random.shuffle(brown_tagged_sents) # permutiraj podatke 
train_sents = brown_tagged_sents[:size]
test_sents = brown_tagged_sents[size:]

# Define regexp tagger patterns
patterns = [
    (r'.*ing$', 'VERB'),  # gerunds
    (r'.*ed$', 'VERB'),   # simple past
    (r'.*es$', 'VERB'),   # 3rd singular present
    (r'.*ould$', 'VERB'), # modals
    (r'.*\'s$', 'NOUN'),  # possessive nouns
    (r'.*s$', 'NOUN'),    # plural nouns
    (r'^-?[0-9]+(.[0-9]+)?$', 'NUM'),  # cardinal numbers
    (r'.*', 'NOUN')       # nouns (default)
]

regexp_tagger = RegexpTagger(patterns)

# Test sentence for demonstration
test_sent = brown_sents[5]
tagged_sent = regexp_tagger.tag(test_sent)

In [23]:
t0 = regexp_tagger # osnovni regexp parser
t1 = nltk.UnigramTagger(train_sents, backoff=t0)
t2 = nltk.BigramTagger(train_sents, backoff=t1)
t3 = nltk.TrigramTagger(train_sents,backoff=t2)


# ispisi primjere
for tok, tag in tagged_sent:
    print(tok,tag)
    
print(f'Preciznost: {t3.accuracy(test_sents)}') # udio podudarajućih vlastitih oznaka sa standardnim 

It NOUN
recommended VERB
that NOUN
Fulton NOUN
legislators NOUN
act NOUN
`` NOUN
to NOUN
have NOUN
these NOUN
laws NOUN
studied VERB
and NOUN
revised VERB
to NOUN
the NOUN
end NOUN
of NOUN
modernizing VERB
and NOUN
improving VERB
them NOUN
'' NOUN
. NOUN
Preciznost: 0.9427684547482782


In [24]:
# matrica zbunjenosti
test_tags = [tag for sent in brown.sents(categories='news')[:10] for (word, tag) in t3.tag(sent)]
gold_tags = [tag for sent in brown.tagged_sents(categories='news',tagset='universal')[:10] for (word, tag) in sent]

cm = nltk.ConfusionMatrix(gold_tags, test_tags)
print(cm)

     |              C     N     P     V |
     |     A  A  A  O  D  O  N  R  P  E |
     |     D  D  D  N  E  U  U  O  R  R |
     |  .  J  P  V  J  T  N  M  N  T  B |
-----+----------------------------------+
   . |<38> .  .  .  .  .  .  .  .  .  . |
 ADJ |  .<19> .  .  .  .  .  .  .  .  . |
 ADP |  .  .<30> .  .  .  .  .  .  2  . |
 ADV |  .  .  . <6> .  .  .  .  .  .  . |
CONJ |  .  .  .  .<10> .  .  .  .  .  . |
 DET |  .  .  .  .  .<39> .  .  .  .  . |
NOUN |  .  .  .  .  .  .<81> .  .  .  . |
 NUM |  .  .  .  .  .  .  . <1> .  .  . |
PRON |  .  .  .  .  .  .  .  . <6> .  . |
 PRT |  .  .  .  .  .  .  .  .  . <3> . |
VERB |  .  .  .  .  .  .  1  .  .  .<48>|
-----+----------------------------------+
(row = reference; col = test)

